# Diabetic Retinopathy Stage Detection
**BSc Computer Science — Computer Vision Module Coursework**

This notebook implements a complete pipeline for detecting and staging Diabetic Retinopathy (DR) using
deep learning (EfficientNetB3 transfer learning) with explainability (Grad-CAM), embedding-based
similar-case retrieval, and a multi-agent clinical decision pipeline.

**Dataset:** Combined DR Dataset (APTOS + IDRiD + Messidor-2 + EyePACS subset)
**Classes (ICDR 0–4 scale):**
- 0: No DR
- 1: Mild NPDR
- 2: Moderate NPDR
- 3: Severe NPDR
- 4: Proliferative DR (PDR)

---
## Pipeline Sections
1. Setup & Data Acquisition
2. Preprocessing
3. Data Augmentation & Class Balancing
4. Train/Validation/Test Split
5. Model — CNN with Transfer Learning (EfficientNetB3)
6. Training Strategy
7. Evaluation
8. Explainability — Grad-CAM
9. Innovation A — Embedding-Based Similar-Case Retrieval
10. Innovation B — Multi-Agent Clinical Decision Pipeline
11. UI — Gradio Interface
---

## Section 1: Setup & Data Acquisition
**Goal:** Install dependencies, configure all project constants in one place (Config class),
verify GPU, download the dataset via the Kaggle API, and load/validate the labels —
including a robust fallback to folder-based labelling if no CSV is found,
corrupt-image filtering, and a class-distribution summary.

In [ ]:
# ============================================================
# SECTION 1.1 — INSTALL DEPENDENCIES
# Run this cell once. Restart the runtime if prompted after install.
# ============================================================
!pip install -q kaggle scikit-learn seaborn pillow opencv-python-headless gradio matplotlib tqdm pandas numpy tensorflow

In [ ]:
# ============================================================
# SECTION 1.2 — GLOBAL CONFIGURATION (single source of truth)
# All magic numbers and tunable hyperparameters live here.
# Modify Config values rather than hunting through code.
# ============================================================

import os
import random
import numpy as np
import tensorflow as tf


class Config:
    """
    Central configuration object.

    All project-wide constants are defined here so they can be changed in
    one place. Keeping constants here also makes experiments reproducible:
    bump SEED and everything downstream uses the new value automatically.
    """

    # --- Reproducibility ---
    SEED: int = 42

    # --- Dataset paths ---
    KAGGLE_DATASET: str = "harsha1289/combined-dr-dataset-aptosidridmessidoreyepacs"
    DATA_DIR: str       = "/content/dr_data"
    TRAIN_DIR: str      = "/content/dr_data/train"
    # CSV filenames to try, in priority order
    LABEL_CANDIDATES: list = ["train.csv", "labels.csv", "trainLabels.csv"]

    # --- Image preprocessing ---
    IMG_SIZE: int          = 224   # EfficientNetB3 default input resolution
    BEN_GRAHAM_SIGMA: int  = 10    # Gaussian blur radius for Ben Graham enhancement
    BEN_GRAHAM_ALPHA: float = 4.0  # weight on original image
    BEN_GRAHAM_BETA: float  = -4.0 # weight on blurred image (subtracted)
    BEN_GRAHAM_GAMMA: float = 128  # additive bias to centre pixel distribution

    # --- Class labels (ICDR 0-4 scale) ---
    CLASS_NAMES: list = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
    NUM_CLASSES: int  = 5

    # --- Train / Val / Test split ---
    TRAIN_RATIO: float = 0.70
    VAL_RATIO: float   = 0.15
    TEST_RATIO: float  = 0.15   # three ratios must sum to 1.0

    # --- Model / Training hyperparameters ---
    BATCH_SIZE: int      = 32
    PHASE1_EPOCHS: int   = 15    # Phase 1: frozen base, train head only
    PHASE2_EPOCHS: int   = 25    # Phase 2: fine-tune top N base layers
    PHASE1_LR: float     = 1e-3  # higher LR is safe when base is frozen
    PHASE2_LR: float     = 1e-5  # very low LR prevents catastrophic forgetting
    DROPOUT_RATE: float  = 0.3   # applied before final softmax
    DENSE_UNITS: int     = 256   # units in intermediate dense layer
    UNFREEZE_TOP_N: int  = 30    # base-model layers to unfreeze in phase 2

    # --- Callbacks ---
    ES_PATIENCE: int     = 5     # early stopping patience (val_loss)
    RLROP_FACTOR: float  = 0.5   # LR reduction factor on plateau
    RLROP_PATIENCE: int  = 3     # patience before reducing LR

    # --- GovernanceAgent threshold ---
    CONFIDENCE_THRESHOLD: float = 0.70  # flag predictions below this confidence

    # --- Output paths ---
    CHECKPOINT_DIR: str  = "/content/checkpoints"
    REPORTS_DIR: str     = "/content/report_images"
    EMBEDDINGS_PATH: str = "/content/embeddings.npz"


def set_all_seeds(seed: int = Config.SEED) -> None:
    """
    Set random seeds for Python, NumPy, and TensorFlow.

    Seeds are set globally so all downstream random operations — data
    shuffling, weight initialisation, augmentation — produce the same
    results across runs, making experiments reproducible.

    Args:
        seed: Integer seed value. Defaults to Config.SEED.
    """
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"[Config] All random seeds set to {seed}.")


set_all_seeds()